# Mercado: Lieferqualität und Kundenservice
 **DABI1 · Projektaufgabe**

Gruppe | Studierende 1 | Studierende 2 | Studierende 3 |
------ | ------ | ------ |-----|
`8` | `Damian Frehar` | `Nils Hahn`|`Maximilian Hutter` |

## 2. Ihr Beratungsauftrag

### 2.1 Daten vorbereiten

#### 2.1 a Datenqualität prüfen

##### Import CSV-Dateien und Python-Bibliotheken

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import pprint as pp
import missingno as msno

In [ ]:
# Dictionary für die DataFrames
dataframes = {}

data_folder = Path("data")

# Alle CSV-Dateien im Ordner durchlaufen
for file in data_folder.glob("*.csv"):
    df = pd.read_csv(file, dtype={"customer_zip_code_prefix": str, "seller_zip_code_prefix": str, "geolocation_zip_code_prefix": str})
    dataframes[file.stem] = df
    print(file)

In [ ]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

##### Orders.csv

In [ ]:
orders = dataframes["orders"]
orders

In [ ]:
orders.dtypes

In [ ]:
orders.head()

###### Datentypen
(Konvertierung in Datatime)

In [ ]:
orders = dataframes["orders"]

datumsspalten = ["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"]

for spalte in datumsspalten:
    orders[spalte] = pd.to_datetime(orders[spalte], format="%Y-%m-%d %H:%M:%S")

orders.dtypes

###### Schlüssel

In [ ]:
schluessel = [
        col for col in orders.columns
        if orders[col].notna().all()
        and orders[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für orders ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für orders ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(orders)

Bei stornierten Bestellungen (canceled) fehlt das Lieferdatum beim Kunden fast immer, meist auch die Übergabe an den Transporteur. Das ist plausibel, weil die Ware nicht verschickt wurde. Ausnahme: 6 stornierte Bestellungen haben trotzdem ein Lieferdatum. Sie wurden vermutlich erst nach der Zustellung storniert, z. B. durch eine Rücksendung.

In [ ]:
orders[orders["order_status"] == "canceled"]

orders["not_approved"] = orders["order_approved_at"].isna()
px.histogram(orders, y="order_status", color="not_approved", width=1000, title="Status vs. Freigabe")

In [ ]:
orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

In [ ]:
(orders["order_status"].eq("canceled") & orders["order_delivered_customer_date"].notna()).sum()

160 Bestellungen haben kein Freigabedatum (order_approved_at). Davon sind 141 canceled und 5 created; das ist plausibel, weil sie nie freigegeben wurden. Auffällig sind 14 Bestellungen mit Status delivered ohne Freigabedatum: Sie wurden geliefert, die Freigabe wurde aber nicht erfasst. Wir behalten sie, weil wir das Freigabedatum für die Merkmale (Lieferdauer, Verspätung) nicht brauchen.

###### Plausible Größenordnungen

In [ ]:
for spalte in datumsspalten:
    print(spalte)
    print(orders[spalte].min())
    print(orders[spalte].max())
    print("")

In [ ]:
orders["delivery_time"] = orders["order_delivered_carrier_date"] - orders["order_purchase_timestamp"]
orders["delivery_time"].describe()

In [ ]:
orders["zeit_bis_uebergabe"] = (orders["order_delivered_carrier_date"]
    - orders["order_purchase_timestamp"]) / pd.Timedelta(hours=1)
orders["zeit_bis_uebergabe"].describe()

In [ ]:
negativ = orders[orders["zeit_bis_uebergabe"] < 0]

px.scatter(negativ, x="zeit_bis_uebergabe",
             title=f"Übergabe an Transporteur vor Kauf ({len(negativ)} Bestellungen)",
             labels={"zeit_bis_uebergabe": "Stunden zwischen Kauf und Übergabe"})

In [ ]:
negativ["zeit_bis_uebergabe"].describe()

In [ ]:
negativ2 = orders[(orders["zeit_bis_uebergabe"] < 0) & (orders["zeit_bis_uebergabe"] > -100)]

px.scatter(negativ2, x="zeit_bis_uebergabe",
             title=f"Übergabe an Transporteur vor Kauf ({len(negativ2)} Bestellungen)",
             labels={"zeit_bis_uebergabe": "Stunden zwischen Kauf und Übergabe"})

In [ ]:
(orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum()

In [ ]:
(orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()

Kategorien von order status

In [ ]:
orders["order_status"].unique()

Fazit:
Datumspalten wurden in Datetime umgewandelt,
Fehlende Werte bei orders werden beibehalten, da die Quantität der fehlenden Daten überschaubar ist, und die Werte aus einem plausiblen Grund fehlen


##### Customers.csv

In [ ]:
customers = dataframes["customers"]

###### Datentypen

In [ ]:
customers.dtypes

In [ ]:
customers.head()

In [ ]:
customers["customer_city"].unique()

In [ ]:
customers["customer_state"].unique()

###### Schlüssel

In [ ]:
schluessel = [
        col for col in customers.columns
        if customers[col].notna().all()
        and customers[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für customers ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für customers ist die Spalte: {schluessel}")

macht sinn weil customer_unique_id ist mehrmals vorhanden da Kunden mehrere Bestellungen machen können

###### Fehlende Werte

In [ ]:
msno.matrix(customers)

keine fehlenden Werte :)

###### Plausible Größenordnungen

In [ ]:
px.histogram(customers, x="customer_state", color="customer_city")

sao paulo macht sinn weil das die größte Stadt ist

##### Items.csv

###### Datentypen

In [ ]:
items = dataframes["items"]

In [ ]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S")

In [ ]:
items.dtypes

In [ ]:
items

###### Schlüssel

In [ ]:
schluessel = [
        col for col in items.columns
        if items[col].notna().all()
        and items[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für items ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für items ist die Spalte: {schluessel}")

In [ ]:
items["order_id"].nunique()

In [ ]:
items["order_key"] = items["order_id"].astype(str) + "_" + items["order_item_id"].astype(str)

In [ ]:
items["order_key"].nunique()

In [ ]:
items["order_key"].duplicated(keep=False)

In [ ]:
items["order_key"].duplicated().sum()

In [ ]:
items[items["order_key"].duplicated(keep=False)].sort_values("order_key")

In [ ]:
items["order_key"].value_counts()[lambda x: x > 1]

In [ ]:
items.duplicated().sum()

In [ ]:
# Schauen was der Umsatz mit den Duplikaten ist
items["price"].sum()

Wir haben 225 Zeilen welche identische Duplikate sind 

In [ ]:
items = items.drop_duplicates()

In [ ]:
schluessel = [
        col for col in items.columns
        if items[col].notna().all()
        and items[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für items ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für items ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(items)

keine fehlenden Werte in items.csv

###### Plausible Größenordnungen

In [ ]:
px.histogram(items, x="price")

In [ ]:
items["price"].describe()

In [ ]:
items["price"].sum()

In [ ]:
#Untersuchen was das teuerste Produkt ist
items.sort_values(by="price", ascending=False)

In [ ]:
products = dataframes["products"]
products[products["product_id"] == "489ae2aa008f021502940f251d4cce7f"]

In [ ]:
px.histogram(items, x="freight_value")

In [ ]:
items['freight_value'].describe()

In [ ]:
(items["freight_value"] == 0).sum()

In [ ]:
items["shipping_limit_date"].max()
items["shipping_limit_date"].min()

In [ ]:
items_merge = pd.merge(items, products, on="product_id")

In [ ]:

px.scatter(items_merge, x="product_weight_g", y="freight_value", title="Zusammenhang zwischen Gewicht und Frachtwert")

##### Sellers.csv

In [ ]:
sellers = dataframes["sellers"]

###### Datentypen

In [ ]:
sellers.dtypes

In [ ]:
sellers.head()

In [ ]:
sellers["seller_city"].unique()

In [ ]:
sellers["seller_state"].unique()

###### Schlüssel

In [ ]:
schluessel = [
    col for col in sellers.columns
    if sellers[col].notna().all()
    and sellers[col].is_unique
]

if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für sellers ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für sellers ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(sellers)

keine Fehlenden Werte

###### Plausible Größenordnungen

In [ ]:
px.histogram(sellers, x="seller_state", color="seller_city")

Sao Paulo ist wieder nummer 1, als größte Stadt

##### Reviews.csv

In [ ]:
reviews = dataframes["reviews"]

###### Datentypen

In [ ]:
reviews.dtypes

In [ ]:
reviews.head()

In [ ]:
datumsspalten = ["review_creation_date", "review_answer_timestamp"]

for spalte in datumsspalten:
    reviews[spalte] = pd.to_datetime(reviews[spalte], format="%Y-%m-%d %H:%M:%S")

reviews.dtypes

In [ ]:
reviews["review_score"].unique()

###### Schlüssel

In [ ]:
schluessel = [
    col for col in reviews.columns
    if reviews[col].notna().all()
    and reviews[col].is_unique
]

if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für reviews ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für reviews ist die Spalte: {schluessel}")

In [ ]:
reviews["review_id"].duplicated().sum()

In [ ]:
(reviews["order_id"].value_counts() > 1).sum() 

814 review_ids kommen mehrfach vor. Dieselbe Bewertung ist also mehreren Bestellungen zugeordnet. 547 Bestellungen haben mehr als eine Bewertung, was 551 überzähligen Zeilen entspricht, weil manche Bestellungen drei Bewertungen haben. Die Tabelle hat deshalb keinen eindeutigen Schlüssel. Für die Aufgaben verwenden wir je Bestellung die letzte Bewertung.

###### Fehlende Werte

In [ ]:
msno.matrix(reviews)

Keine Fehlenden Werte

###### Plausible Größenordnungen

In [ ]:
for spalte in datumsspalten:
    print(spalte)
    print(reviews[spalte].min())
    print(reviews[spalte].max())
    print("")

In [ ]:
reviews["review_score"].unique()

Bewertungen von 1-5, über zeitraum von 2016-2018 passt

##### Products.csv

###### Datentypen

In [ ]:
products = dataframes["products"]

In [ ]:
products

In [ ]:
products.dtypes

###### Schlüssel

In [ ]:
schluessel = [
        col for col in products.columns
        if products[col].notna().all()
        and products[col].is_unique
    ]

if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für products ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für products ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(products)

Products_category_name kann nicht ersetzt werden, da wir die Kategorien für die neuen product_ids nicht kennen.
Product_weight_g ist ein individueller Value, weshalb er auch nicht ersetzt werden kann (kann für Berechnungen, bei denen man für alles ein Value braucht, mit dem Median ersetzt werden).
Products_photos_qty ist immer ein Null-Value, wenn products_category_name ein Null-Value ist.

In [ ]:
products.sort_values(by = "product_photos_qty", ascending =  False)

In [ ]:
products["product_category_name"].nunique()

In [ ]:
categories = dataframes["categories"]
categories.sort_values(by="product_category_name_english", ascending = False)
categories["product_category_name"].unique()

In [ ]:
products["product_category_name"] = products["product_category_name"].str.lower()
products["product_category_name"].nunique()

products["product_category_name"].sort_values().unique()

In [ ]:
products["product_category_name"] = products["product_category_name"].str.strip()
products["product_category_name"].nunique()

In [ ]:
from difflib import get_close_matches

def korrigiere_kategorien(werte, gueltige_kategorien, mindest_score=0.8):
    referenz = set(gueltige_kategorien.dropna())
    normalisiert = werte

    ersetzungen = {}
    nicht_zugeordnet = []
    for wert in normalisiert.dropna().unique():
        if wert in referenz:
            continue
        treffer = get_close_matches(wert, referenz, n=1, cutoff=mindest_score)
        if treffer:
            ersetzungen[wert] = treffer[0]
        else:
            nicht_zugeordnet.append(wert)

    return normalisiert.replace(ersetzungen), ersetzungen, nicht_zugeordnet

In [ ]:
products["kategorie_clean"], ersetzungen, rest = korrigiere_kategorien(
    products["product_category_name"],
    categories["product_category_name"], 0.8
)

print(products["kategorie_clean"].nunique())
print(ersetzungen)
print(rest)

In [ ]:
categories[categories["product_category_name_english"].isna()]

In [ ]:
categories.loc[60, "product_category_name_english"]= "pc_gamer"
categories.loc[65, "product_category_name_english"]= "portable_kitchen_food_processors"

In [ ]:

products = products.merge(
    categories, left_on="kategorie_clean", right_on="product_category_name"
)

Products.csv ist ein gutes Beispiel, da man direkt anhand der Kennzahl uniquer `product_category_name` sehen kann wie wir schrittweise unsere Berreinigung durchführen.

Wir gehen dabei von 236 auf 73, durch das enfernen von fehlerhaften Leerzeichen und anderen Schreibfehlern

###### Plausible Größenordnungen

In [ ]:
products["product_weight_kg"] = products["product_weight_g"]/1000

In [ ]:
px.scatter(products, y="product_weight_kg", color="product_category_name_english")


In [ ]:
cutted = products[products["product_weight_kg"] < 100]
px.scatter(cutted, y="product_weight_kg", color="product_category_name_english")


##### Zusammenfassung: Befunde und Entscheidungen

| Tabelle | Befund | Entscheidung | Begründung |
|---|---|---|---|
| alle | IDs und PLZ-Präfixe sind Text mit führenden Nullen | PLZ beim Einlesen als `str` | Führende Nullen würden als Zahl verloren gehen |
| orders, items, reviews | Zeitstempel als Text eingelesen | Umwandlung mit `pd.to_datetime` | Nötig für Lieferdauer, Verspätung und Zeitvergleiche |
| orders | 8 Statuswerte, nur `delivered` hat vollständige Lieferdaten | Für 3.1–3.3 nur `delivered` mit beiden Lieferterminen | Lieferdauer und Verspätung sind nur bei gelieferten Bestellungen definiert |
| orders | 160 Bestellungen ohne Freigabedatum (141 canceled, 5 created, 14 delivered) | Behalten | Plausibel erklärbar; Freigabedatum wird für die Merkmale nicht benötigt |
| orders | 6 stornierte Bestellungen mit Lieferdatum, 8 gelieferte ohne Lieferdatum | Behalten, fallen durch den Filter heraus | Sonderfälle (z. B. Storno nach Zustellung); kein Einfluss auf die Analysen |
| orders | 166 Bestellungen mit Übergabe an den Transporteur vor Kaufzeitpunkt, 23 mit Lieferung vor Übergabe | Behalten | Meist Abweichungen unter 1 Stunde → Ungenauigkeit der Zeitstempel; Übergabedatum wird nicht verwendet |
| orders | Lieferdauern bis ca. 210 Tage | Behalten | Lange Lieferungen sind möglich und für die Logistik relevant; ungewöhnlich ≠ Fehler |
| customers | `customer_id` eindeutig, `customer_unique_id` mehrfach | `customer_id` als Schlüssel | Ein Kunde kann mehrere Bestellungen haben (Wiederholungskäufer) |
| items | 225 vollständig doppelte Zeilen (gleiche `order_id` + `order_item_id`) | Entfernt mit `drop_duplicates()` | Echte Duplikate würden Warenwert und Positionszahl verfälschen |
| items | 383 Positionen mit Frachtkosten 0 | Behalten | Kostenloser Versand ist plausibel |
| items | Sehr hohe Einzelpreise (bis 6.735 BRL) | Behalten | Zum Produkt passend geprüft, plausibel |
| sellers | `seller_city` uneinheitlich geschrieben (z. B. „sao paulo - sp“) | Nicht bereinigt | Für die Fragen wird nur der Bundesstaat genutzt, und der ist sauber |
| reviews | 814 doppelte `review_id`, 547 Bestellungen mit mehreren Bewertungen | Je Bestellung die letzte Bewertung (nach `review_answer_timestamp`, dann `review_id`) | So zählt jede Bestellung genau einmal |
| products | Kategorienamen uneinheitlich: Großschreibung, Leerzeichen, Tippfehler (236 Werte) | `lower` + `strip` + Abgleich mit `categories.csv` → 73 Kategorien | Sonst würden gleiche Kategorien getrennt gezählt |
| products | Fehlende Kategorie, Gewicht und Fotoanzahl bei einzelnen Produkten | Nicht ersetzt | Werte lassen sich nicht sinnvoll schätzen; für die Pflichtfragen nicht nötig |
| products | 32 Produkte mit ≥ 100 kg bei kleinen Maßen (bis 5.850 kg) | Gewicht nicht verwenden bzw. als fehlend markieren | Vermutlich Erfassungsfehler (Faktor 1000); Gewicht wird in 3.1–3.3 nicht benötigt |
| categories | 2 Kategorien ohne englische Übersetzung | Übersetzung manuell ergänzen | Sonst fehlen sie in Auswertungen nach englischem Kategorienamen |
| Bestelltabelle | Grundgesamtheit für 3.1–3.3 | Filter: delivered + beide Liefertermine + Bewertung 1–5 → **95.824 Bestellungen** | Vorgabe der Aufgabe; jede Bestellung genau einmal (`order_id` eindeutig) |

**Auswirkung einer Bereinigung an einer Kennzahl:** Anzahl verschiedener Produktkategorien 236 → 73 (lower/strip/Tippfehler). Entfernen der 225 doppelten Positionen senkt den Gesamtumsatz von 13.620.521 auf 13.591.644 BRL.

#### 2.1 b Bestelltabelle (eine Zeile je Bestellung)

In [ ]:
payments = dataframes["payments"]
payments.head()

##### Item & Seller Merge

In [ ]:
positionen = items.merge(sellers, on="seller_id", how="left")

##### Positionen je Bestellung zusammenfassen

In [ ]:
items_agg = positionen.groupby("order_id").agg(
    anzahl_positionen=("order_item_id", "count"),
    anzahl_haendler=("seller_id", "nunique"),
    warenwert=("price", "sum"),
    frachtkosten=("freight_value", "sum"),
    seller_states=("seller_state", lambda s: sorted(set(s))),
).reset_index()

##### Payments je Bestellung zusammenfassen

In [ ]:
payments_agg = payments.groupby("order_id").agg(
    number_of_payments=("payment_sequential", "count"),
    payment_type=("payment_type", "nunique"),
    sum_of_rates=("payment_installments", "sum"),
    payment_value=("payment_value", "sum"),
    payment_types=("payment_type", lambda s: sorted(set(s))),
).reset_index().sort_values(by="number_of_payments", ascending=False)

##### Letzte Bewertung je Bestellung

In [ ]:
reviews_last = (
    reviews
    .sort_values(["order_id", "review_answer_timestamp", "review_id"])
    .drop_duplicates("order_id", keep="last")
)

##### Alles an orders hängen

In [ ]:
bestellungen = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(reviews_last[["order_id", "review_score", "review_answer_timestamp"]],
           on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
)

len(orders), len(bestellungen), bestellungen["order_id"].is_unique

##### Auf bestellungen mit tatsächlichem und zugesagtem Liefertermin & letzter gültiger Bewertung zwischen 1-5 limitieren

In [ ]:
bestellungen.head()

In [ ]:
bestellungen = bestellungen[
    (bestellungen["order_status"] == "delivered")
    & bestellungen["order_delivered_customer_date"].notna()
    & bestellungen["order_estimated_delivery_date"].notna()
    & bestellungen["review_score"].between(1, 5)
].copy()

In [ ]:
len(orders), len(bestellungen), bestellungen["order_id"].is_unique

Es bleiben noch 95824 übrig

#### 2.1 c Merkmale

In [ ]:
bestellungen.head()

##### Schlechte Review (score höchstens 2)

In [ ]:
bestellungen["bad_review"] = bestellungen["review_score"] <= 2

bestellungen["bad_review"]

##### Lieferdauer

In [ ]:
bestellungen["delivery_time"] = (bestellungen["order_delivered_customer_date"]
    - bestellungen["order_purchase_timestamp"]) / pd.Timedelta(days=1)

bins = [-float("inf"), 7, 14, 21, float("inf")]

labels = ["≤ 7 Tage", "> 7 bis 14 Tage", "> 14 bis 21 Tage", "> 21 Tage"]

bestellungen["delivery_group"] = pd.cut(
    bestellungen["delivery_time"],
    bins=bins,
    labels=labels
)

bestellungen["delivery_group"].value_counts().sort_index()

##### Verspätung

In [ ]:
bestellungen["order_delay"] = (bestellungen["order_delivered_customer_date"].dt.normalize() - bestellungen["order_estimated_delivery_date"]).dt.days
bestellungen["verspaetet"] = bestellungen["order_delay"] > 0
bestellungen["order_delay"].describe()

In [ ]:
bestellungen.head()

##### Versand zwischen Bundesstaaten

In [ ]:
bestellungen["zwischen_staaten"] = [
    any(s != kunde for s in haendler) if isinstance(haendler, list) else pd.NA
    for kunde, haendler in zip(bestellungen["customer_state"], bestellungen["seller_states"])
]
bestellungen["zwischen_staaten"].value_counts(dropna=False)

##### Warenkorb / Fracht 

In [ ]:
bestellungen["delivery_cost_as_percentage_of_total_cost"] = bestellungen["frachtkosten"] / (bestellungen["warenwert"] + bestellungen["frachtkosten"])
bestellungen["delivery_cost_as_percentage_of_total_cost"]

In [ ]:
bestellungen[["anzahl_positionen","warenwert","delivery_cost_as_percentage_of_total_cost"]]

##### Eigenes Merkmal: Wiederholungskunde

In [ ]:
bestellungen["wiederholungskunde"] = bestellungen.groupby("customer_unique_id")["order_id"].transform("count") > 1
px.histogram(bestellungen, x="wiederholungskunde", width=400)

##### Ab wann verfügbar 

|Merkmal |	verfügbar ab|
|---|---|
|Schlechte Bewertung|	nach der Bewertung|
|Lieferdauer, Verspätung	|bei Lieferung|
Zwischen Staaten, Warenkorb, Frachtanteil|beim Kauf
Wiederholungskunde | beim Kauf


### 2.2 Data Stories

#### Logistikleitung

#### Serviceleitung

## 3. Stakeholderfragen

### 3.1 Versand und Bewertungen

### 3.2 Automatische Nachricht

In [ ]:
service_observation = dataframes["service_observation"]
service_observation["service_start_at"] = pd.to_datetime(service_observation["service_start_at"], format="%Y-%m-%d %H:%M:%S")
service_observation.head()

In [ ]:
bestellungen.head().T

In [ ]:
verspaetung = pd.DataFrame({
    "order_id": bestellungen["order_id"],
    "verspaetung_tage": (
        bestellungen["order_delivered_customer_date"].dt.normalize() - bestellungen["order_estimated_delivery_date"]).dt.days,
})
verspaetung["verspaetet"] = verspaetung["verspaetung_tage"] > 0

verspaetung.head()

In [ ]:
order_feedback =pd.merge(service_observation, verspaetung, on="order_id", how="left").copy()
order_feedback.head()

a) Vergleichen Sie den Anteil der Bestellungen mit mindestens einer Beschwerde: mit automatischer
Nachricht gegenüber ohne Nachricht. Jede Bestellung zählt als ein Fall.

In [ ]:
pd.crosstab(
    order_feedback["automatic_notice"],
    order_feedback["complaint_7d"] >= 1,
    margins=True,
    normalize="index",
)*100

In [ ]:
pd.crosstab(order_feedback["automatic_notice"], order_feedback["complaint_7d"], margins=True)

Bei den Kunden die eine `automatic_notice` bekommen haben sich 17,1 % beschwert und bei denen die keine Bekommen haben nur 7,4%

$\rightarrow$ Also 9,7% mehr 

b) Hängt der Nachrichtenversand mit Verspätungen zusammen? Hängen Beschwerden mit Verspätungen
zusammen?

In [ ]:
pd.crosstab(
    order_feedback["verspaetet"],
    order_feedback["automatic_notice"] >= 1,
    normalize="index"
) *100


In [ ]:
pd.crosstab(
    order_feedback["verspaetet"],
    order_feedback["automatic_notice"] >= 1,
     margins=True
)

Die Werte der Kreuztabelle deuten darauf hin, dass der Nachrichtenversand mit der verspäteten Lieferzeit zusammenhängt.

In [ ]:
pd.crosstab(
    order_feedback["verspaetet"],
    order_feedback["complaint_7d"] >= 1,
    normalize="index"
) *100

In [ ]:
pd.crosstab(
    order_feedback["verspaetet"],
    order_feedback["complaint_7d"] >= 1,
    margins=True
)

Die Werte der Kreuztabelle deuten darauf hin, dass die Beschwerde mit der verspäteten Lieferzeit zusammenhängt.

### 3.3 Persönliche Kundenbetreuung


# Zusammenarbeit & KI

  - wer macht was (Damian / Nils / Max)
  - wie wird zusammengeführt (wer pflegt das Haupt-Notebook)
  - wie prüft wir gegenseitig
  - wie wird KI genutzt und wie werden die Vorschläge geprüft und dokumentie